<a href="https://colab.research.google.com/github/vs-tries-to-code/flyrank-ml-starter/blob/week3/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of Analysis (The Grain**) :
One row = One unique URL/Page (page_id or url) at a specific snapshot point in time (e.g., snapshot as of 2026-03-31).

**Time Window :**
The time window of analysis is chosen to be a mid panel month - March 2026

**Tables to be used :**
```dim_content``` and ```fact_content_daily_performance```

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Bucket | Fields | Reason |
| :--- | :--- | :--- |
| Feature | gsc_impressions, gsc_clicks, ga4_engaged_sessions, freshness (content_updated_date - content_created_date) | Observed search measurements, logged as they happen -> knowable at the decision moment|
| Context | content_hash_id, client_hash_id, word_count  | Identification attributes and content length |
| Label/proxy| label is_declining, a binary flag indicating decline | Computed using change in impressions in the second half of a month compared with the first|
| Excluded | health_score, priority_score, needs_ctr_fix, is_quick_win, action_type (not present in this release, named for completeness); | To prevent data leakage by supplying the metrics to be predicted

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [2]:
import duckdb
from google.colab import userdata

# 1. Retrieve the Hugging Face token you saved in Colab Secrets
hf_token = userdata.get('warehouse')

# 2. Connect to DuckDB
con = duckdb.connect()

# 3. Create the Hugging Face secret in DuckDB dynamically
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}');")



**Counting distinct grain keys**

In [3]:
q1 = """
SELECT COUNT(*) AS 'Entries in 03-2026'
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet')
WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31';
"""
result = con.sql(q1)
print(result)

┌────────────────────┐
│ Entries in 03-2026 │
│       int64        │
├────────────────────┤
│            9841378 │
└────────────────────┘



**The number of rows which survive after removing deleted pages**

The pages which survive are those which are published and not deleted, and the ones for which gsc data is available, otherwise the gsc clicks and impressions cannot be fed as features.

In [5]:
q2 = f"""SELECT
    COUNT(*) AS total_candidates,
    COUNT(CASE WHEN d.is_published IS TRUE AND d.is_deleted IS NOT TRUE AND f.gsc_data_available IS TRUE THEN 1 END) AS surviving_published_rows,
    ROUND(
        100.0 * COUNT(CASE WHEN d.is_published IS TRUE AND d.is_deleted IS NOT TRUE AND (f.gsc_data_available IS TRUE or f.ga4_data_available is TRUE) THEN 1 END) / COUNT(*),
        2
    ) AS survival_percentage
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet') f
LEFT JOIN read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet') d
    ON f.content_hash_id = d.content_hash_id;"""

result = con.sql(q2)
print(result)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────┬──────────────────────────┬─────────────────────┐
│ total_candidates │ surviving_published_rows │ survival_percentage │
│      int64       │          int64           │       double        │
├──────────────────┼──────────────────────────┼─────────────────────┤
│          9841378 │                  3609108 │               37.15 │
└──────────────────┴──────────────────────────┴─────────────────────┘



## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

1. The data does not account for algorithm Updates (SERP Volatility), which may or may not lead to the decline of a page.

2. Internal metrics like search_volume and competition_level exists, but no live competitor data. It cannot be determined if a competitor published a far superior piece of content or launched a massive backlink campaign that outranked a page.

3. Word count and character count are proxy metrics, they do not equal quality.

4. The dataset lacks post-click behavior like bounce rate, dwell time, conversion rate, or scroll depth, and these factors cannot be predicted.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.